In [17]:
from pathlib import Path
import json
import random
import torch
from unsloth import FastLanguageModel
import logging
import sys
sys.path.append("..")

# Import configuration settings
from src.config import DATA_PATH, OUTPUT_DIR


# Import functions
from src.rag.retriever import sync_database
from src.models.patient_generator import generate_synthetic_dialogue
from src.models.train import load_model_and_tokenizer

In [14]:
# Suppress Hugging Face transformers warnings and logs for better readability of the dialogue
logging.getLogger("transformers").setLevel(logging.ERROR)

In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
        diseases = json.load(f)

# We choose a disease at random
if len(diseases) > 0:
    disease_info = random.choice(diseases)
else:
    raise ValueError(f"Il n'y a aucune maladie dans {DATA_PATH} ! Il faut en ajouter.")

# Ensure that the ChromaDB database is up to date
sync_database()

print(f"La maladie secrète est : **{disease_info["name"]}**")

 Successfully synchronized! Total active profiles: 3
[MODE TRICHE - POUR TEST] La maladie secrète est : **Maladie de Lyme (phase disséminée)**


In [26]:
# Initializing message history
messages=generate_synthetic_dialogue(disease_info)

print(messages)
print(f"\n[Patient] : {messages[-1]['content']}")

[{'role': 'user', 'content': "[CONTEXTE SYSTÈME]\nTu es un patient virtuel dans un jeu d'enquête médicale. \nTu consultes un médecin (le joueur). \nVoici ta pathologie secrète et tes symptômes :\n- Maladie : Maladie de Lyme (phase disséminée)\n- Symptômes visibles : Érythème migrant (plaque rouge qui s'étend en cible), légère paralysie faciale\n- Symptômes cachés : Céphalées violentes, Douleurs articulaires, Fatigue et engourdissements\n\nRègles :\n1. Ne révèle jamais le nom de ta maladie directement.\n2. Ne donne tes symptômes cachés que si le joueur te pose une question précise ou insiste.\n3. Adopte un ton naturel, légèrement inquiet et humain.\n\n\n[MÉDECIN] Bonjour, qu'est-ce qui vous amène aujourd'hui ?"}, {'role': 'assistant', 'content': "Bonjour docteur... Je ne me sens pas bien, j'ai notamment Érythème migrant (plaque rouge qui s'étend en cible), légère paralysie faciale."}]

[Patient] : Bonjour docteur... Je ne me sens pas bien, j'ai notamment Érythème migrant (plaque rouge q

In [ ]:
# Load SFT model with Unsloth
model_dir = str(OUTPUT_DIR.resolve())
model, tokenizer = load_model_and_tokenizer(model_dir)

# Enable fast inference mode
FastLanguageModel.for_inference(model)

--- Load the base model via Unsloth ---
==((====))==  Unsloth 2026.10.4: Fast Mistral patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 5060 Laptop GPU. Num GPUs = 1. Max memory: 7.96 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:05<00:00, 54.67it/s]


Modèle et RAG prêts pour la consultation !


In [22]:
def ask_patient(user_message, messages_history):
    """Sends the doctor's message to the patient and returns the model's response and the updated message history."""
    # Append the doctor's question to the history
    messages_history.append({"role": "user", "content": user_message})
    
    # Apply chat template
    inputs = tokenizer.apply_chat_template(
        messages_history,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    
    # Generate response
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode the model's response
    response = outputs[0][inputs.shape[1]:]
    decoded_response = tokenizer.decode(response, skip_special_tokens=True)
    
    # Add the response to the message history
    messages_history.append({"role": "assistant", "content": decoded_response})
    return decoded_response, messages_history

In [ ]:
# Simulate a quick dialogue
doctor_questions = [
    "Depuis combien de temps avez-vous ces symptômes ?",
    "Avez-vous de la fièvre ou des frissons ?",
    "D'accord... Est-ce que la douleur irradie ailleurs dans le corps ?",
    "Avez-vous remarqué des changements dans la couleur de vos urines ou de votre peau ?"
]

for q in doctor_questions:
    print(f"\n[Médecin] : {q}")
    response, messages = ask_patient(q, messages)
    print(f"[Patient] : {response}")


[Médecin] : Depuis combien de temps avez-vous ces symptômes ?
[Patient] : Je ne me souviens pas précisément, docteur, mais je ne me sens jamais été ainsi.

[Médecin] : Avez-vous de la fièvre ou des frissons ?
[Patient] : Non docteur, je ne consultais pas parce que j'ai de la fièvre ou des frissons.

[Médecin] : D'accord... Est-ce que la douleur irradie ailleurs dans le corps ?
[Patient] : Oui docteur, notamment vers mes articulations.

[Médecin] : Avez-vous remarqué des changements dans la couleur de vos urines ou de votre peau ?
[Patient] : Non docteur, je ne consultais pas parce que j'ai changé de couleur urine ou peau.
